# KG catch-up + intent chat session -> turns + knowledge graph

A new process flow on top of **[`kg_intent_chat.ipynb`](kg_intent_chat.ipynb)**: before the live
chat starts, the agent first figures out **how long it's been since it last spoke with this
human, and what they'd talked about back then**, using
**[`gaps_from_kg/get_temporal_containers.py`](../src/cltl/gaps_from_kg/get_temporal_containers.py)**
-- a different query layer over the same GraphDB repository (via `cltl.brain.LongTermMemory`)
than `kg_gap_finder.py`/`intent_gap_finder.py`'s rdflib/SPARQL-endpoint queries, which is what the
rest of the per-turn flow below still uses unchanged.

The new opening phase (see **[`catch_up_from_kg.py`](catch_up_from_kg.py)** for the full
implementation):

1. **`find_last_conversation_date()`** -- when did the agent and this human last speak, at all?
   Falls back to a fixed default if the KG has no record of them yet.
2. **`find_catch_up_topics()`** -- for every activity/condition type the per-turn intent flow
   already knows how to ask about (`chat_sessions.DEFAULT_GAP_ACTIVITY_TYPES` -- exercise, diet,
   sleep, symptoms, medication, ...), is there real HISTORY for this human (something from before
   that last conversation)? Those are the topics worth catching up on, most-recently-discussed
   first.
3. **`render_opening_question()`** -- the LLM phrases ONE natural opening turn naming how long
   it's been and inviting the human to share what's happened since, name-checking the two most
   relevant remembered topics as concrete memory prompts.
4. **`CatchUpQueue`** -- the *remaining* topics, asked about one at a time, but only whenever the
   ordinary per-turn flow itself has nothing else to ask (see next point) -- via
   **`wrap_agent_fn_with_catch_up()`**, which wraps the plain default `agent_fn` so a queued
   catch-up question is asked instead of it, until the queue runs out.

Everything else is **exactly** `kg_intent_chat.ipynb`'s own per-turn flow, unchanged: every turn
still runs through `LLM_EventExtraction` -> `populate_ekg_from_annotations()`, and the moment the
human mentions any NEW activity or condition -- whether in answer to the opening question, a
queued catch-up question, or anything else -- `KgIntentChatSession` looks it up against
`intents/*.json` (`intent_gap_finder.next_intent_gap()`) and asks its own follow-up question about
it, same as ever. The catch-up flow only ever supplies the *opening* turn and fills in for the
*default* LLM reply -- it never competes with an actual intent-driven gap question.

**Before running this:** same requirements as `kg_intent_chat.ipynb` -- `OPENAI_API_KEY` set, and
a reachable knowledge-graph SPARQL endpoint (e.g. a local GraphDB `event_sandbox` repository) at
`KG_ADDRESS` below, ideally one that already has some history for `HUMAN` in it (otherwise there's
nothing to catch up on, and this degrades to a plain "what's new?" opener with an empty
`CatchUpQueue`).

In [ ]:
import time
from datetime import datetime, timedelta

from chat_sessions import KgIntentChatSession, openai_agent, save_turns
import catch_up_from_kg as catch_up

KG_ADDRESS = "http://localhost:7200/repositories/event_sandbox"
KG_LOG_DIR = "kg_logs"

# Directory of intent *.json files -- None auto-discovers the project's own intents/ folder (see
# intent_gap_finder.load_intents()/_default_intents_dir()).
INTENTS_DIR = None

HUMAN = "Mehmet"

# A FRESH id every run, not a fixed constant -- see kg_intent_chat.ipynb's own CHAT_ID cell for
# why (reusing one across runs collides every run's activities on the same subject URIs).
CHAT_ID = int(time.time())

# "Now", for both the catch-up queries below and every activity pushed to the graph this run.
CURRENT_DATE = datetime.now()

# Used only if the KG has no earlier conversation with HUMAN on record at all (e.g. a brand new
# human, or a fresh/empty graph) -- how far back to assume the last (nonexistent) conversation
# was, so there's still a sensible "it's been N days" opener instead of a crash.
FALLBACK_LAST_CONVERSATION_DATE = CURRENT_DATE - timedelta(days=7)


## Step 1: how long has it been, and what's worth catching up on?

Connects to the same KG `KG_ADDRESS` points at (via `cltl.brain.LongTermMemory` this time, not
rdflib) and runs the two `gaps_from_kg` queries described above. Nothing is written to the graph
here -- this step only reads.

In [ ]:
brain = catch_up.connect_brain(KG_ADDRESS, log_dir=KG_LOG_DIR)

last_conversation_date = catch_up.find_last_conversation_date(
    HUMAN, brain, CURRENT_DATE, FALLBACK_LAST_CONVERSATION_DATE
)
catch_up_topics = catch_up.find_catch_up_topics(brain, CURRENT_DATE, last_conversation_date)

print(f"Last conversation with {HUMAN}: {last_conversation_date} "
      f"({(CURRENT_DATE.date() - last_conversation_date.date()).days} day(s) ago)")
print(f"Found {len(catch_up_topics)} catch-up topic(s):")
for topic in catch_up_topics:
    print(f"  - {topic['activity_type']}: {topic['history_count']} past mention(s), "
          f"most recently \"{topic['latest_label']}\" on {topic['latest_date']}")


## Step 2: open the chat

`render_opening_question()` turns the findings above into the agent's very first turn (recorded
via `open_with()`, same as `kg_chat_session.ipynb`'s own opening-greeting cell -- see there for why
it's a real, annotated-and-pushed turn rather than just printed text). The two topics it names are
dropped from `catch_up_topics` before building the `CatchUpQueue` so they're never asked about
twice.

`wrap_agent_fn_with_catch_up()` then wraps the plain `openai_agent()` default reply function: for
as long as the queue still has topics left, a reply that would otherwise be the generic default
LLM one asks about the next queued topic instead -- see `catch_up_from_kg.py`'s own module
docstring for exactly when that fires versus an ordinary intent-driven gap question.

In [ ]:
from kg_chat_gui import run_gui

LEAD_TOPICS = 2
opening_question = catch_up.render_opening_question(
    HUMAN, CURRENT_DATE, last_conversation_date, catch_up_topics, lead_topics=LEAD_TOPICS
)
catch_up_queue = catch_up.CatchUpQueue(catch_up_topics[LEAD_TOPICS:], human=HUMAN)
agent_fn = catch_up.wrap_agent_fn_with_catch_up(openai_agent(), catch_up_queue)

kg_session = KgIntentChatSession(
    chat=CHAT_ID,
    human=HUMAN,
    kg_address=KG_ADDRESS,
    log_dir=KG_LOG_DIR,
    intents_dir=INTENTS_DIR,
    agent_fn=agent_fn,
)
print(f"Loaded {len(kg_session.intents)} intent(s) covering activity types: "
      f"{sorted({t for intent in kg_session.intents for t in intent.get('activity_types', [])})}")
print(f"{len(catch_up_queue)} catch-up topic(s) still queued after the opening turn: "
      f"{[t['activity_type'] for t in catch_up_queue._queue]}")

kg_session.open_with(opening_question)
kg_turns = run_gui(kg_session)


Inspect what was extracted, pushed, and where each agent reply came from:

In [ ]:
print(f"{len(kg_session.turns)} turns, {len(kg_session.annotations)} annotated, "
      f"{len(kg_session.kg_pushes)} pushes to the knowledge graph")
print("reply sources:", kg_session.reply_sources)
print(f"catch-up topics actually asked about: {[t['activity_type'] for t in catch_up_queue.asked]}")
kg_session.kg_pushes


`kg_session.turn_log` has the same per-turn breakdown `kg_intent_chat.ipynb` prints live: what
each turn pushed, which intent (if any) matched, and which requirement its reply was about
(`None` for a default or catch-up-driven reply -- both are indistinguishable from `turn_log`'s own
point of view, since neither comes from `intent_gap_finder`; `kg_session.reply_sources`/
`catch_up_queue.asked` above are what tell them apart).

In [ ]:
kg_session.turn_log

In [ ]:
save_turns(kg_session.turns, "kg_catchup_intent_turns.json")